# Exploratory Data Analysis: Indian Equity Market Universe (2009–2024)

> **CONFIDENTIAL** — Zetheta Algorithms Private Limited  
> **Project:** Bayesian Regime Detection Engine for Equity Direction Forecasting  
> **Snapshot ID:** `snap_phase1_market_data_c6c46ed7b46c`  

This notebook analyzes approximately 15 years of daily empirical market data across Indian equities, midcaps, broad market breadth, volatility (India VIX), and currency (USD/INR).

In [1]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src.data.snapshot import DataSnapshotEngine
from src.utils.config import get_project_root

root = get_project_root()
engine = DataSnapshotEngine()
manifest_file = root / "data" / "manifest.json"
with open(manifest_file, "r") as f:
    manifest = json.load(f)
latest_snapshot = manifest["latest_snapshot"]
print(f"Active Snapshot ID: {latest_snapshot}")
print(f"Cryptographic Verification: {engine.verify_snapshot_integrity(latest_snapshot)}")

## 1. Data Quality and Completeness Audit

In [2]:
quality_df = pd.read_csv(root / "reports" / "tables" / "data_quality_summary.csv")
quality_df[["series_id", "total_records", "missing_percentage", "duplicate_count", "is_date_strictly_monotonic", "status"]]

## 2. Empirical Return Moments & Non-Gaussian Fat Tails

In [3]:
stats_df = pd.read_csv(root / "reports" / "tables" / "eda_return_statistics.csv")
stats_df[["series", "annualized_return", "annualized_volatility", "skewness", "excess_kurtosis", "is_gaussian"]]

## 3. Normalized Price Paths & Breadth Dynamics (2009–2024)

In [4]:
data, meta = engine.load_snapshot(latest_snapshot)
nifty = data["NIFTY_50"]["close"].dropna()
midcap = data["NIFTY_MIDCAP_50"]["close"].dropna()
vix = data["INDIA_VIX"]["close"].dropna()
usdinr = data["USD_INR"]["close"].dropna()

common_idx = nifty.index.intersection(midcap.index).intersection(vix.index).intersection(usdinr.index)
df_aligned = pd.DataFrame({
    "Nifty 50": nifty.loc[common_idx],
    "Nifty Midcap 50": midcap.loc[common_idx],
    "USD/INR": usdinr.loc[common_idx],
}, index=common_idx)

rebased = df_aligned / df_aligned.iloc[0] * 100.0
fig, ax = plt.subplots(figsize=(12, 5))
rebased.plot(ax=ax)
ax.set_title("Normalized Equity & FX Paths (2009-2024)")
ax.set_ylabel("Index (Rebased to 100)")
plt.show()

## 4. Key Quantitative Insights for Regime Modeling

1. **Severe Excess Kurtosis**: Nifty 50 exhibits excess kurtosis of **19.04**, while USD/INR shows **22.03**. Standard Gaussian emission models will understate tail risks; Student-t mixtures with degrees-of-freedom $\nu \in [3, 8]$ are strongly indicated.
2. **Midcap Asymmetric Negative Skew**: Midcap equities exhibit significant negative skewness (**-0.77**), reflecting severe liquidity withdrawals during Risk-Off transitions.
3. **Negative Correlation Clustering**: Correlation between Nifty 50 and India VIX remains strongly negative (-0.65 to -0.80), but decouples during post-shock stabilization when VIX compresses rapidly while equity prices consolidate.